# 20 — Fuzzy target membership

This notebook presents the bounded fuzzy-membership experiment. Training labels receive triangular overlap with immediately adjacent condition classes, while inner stopping, outer validation and final decisions retain the original crisp three-class target.

**Result:** the closest candidate uses 2.5% fuzzy overlap for XGBoost only and retains the hard-label Random Forest. It reaches **81.534%**, 0.090 points below baseline, with zero fold wins. Hard-label training remains selected.

## Course-aligned lifecycle

| Step | Application |
| --- | --- |
| 1. Define the goal and scope | Test whether overlapping condition memberships improve crisp three-class accuracy. |
| 2. Gather the data | Reuse validated labelled data and the accepted hard-label baseline. |
| 3. Explore the data | Compare effective membership, accuracy, class recall, confusion and probability quality. |
| 4. Clean and preprocess the data | Fit established preprocessing on original source rows inside each training partition. |
| 5. Select and engineer features | Keep the accepted feature policy unchanged. |
| 6. Define the machine-learning task | Train on triangular fuzzy memberships; defuzzify by maximum probability and score crisp labels. |
| 7. Partition the data | Reuse five frozen folds; create fuzzy memberships only inside training partitions. |
| 8. Select and train candidate methods | Refit XGBoost and Random Forest at four fixed overlaps, reconstruct the vote and cross one fuzzy component at a time. |
| 9. Evaluate and interpret the results | Apply the accepted promotion gate and inspect probability as well as hard-label metrics. |
| 10. Deploy and iterate | Not applicable: no candidate passes. Keep the local test closed and retain hard labels. |

In [ ]:
from pathlib import Path
import sys
import joblib
from IPython.display import display

STAGE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = STAGE_DIR / 'src'
PROJECT_DIR = STAGE_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

result = joblib.load(
    PROJECT_DIR / '.runtime' / 'fuzzy-target-screen'
    / 'fuzzy-target-screen.joblib'
)
result['membership_summary']

## Membership definition

For raw adjacent overlap $\lambda$, the observed class receives membership one and each immediate neighbour receives $\lambda$; each row is then normalised to total membership one. Functional and non-functional are not direct neighbours.

At $\lambda=0.05$, a functional row becomes $(0.9524, 0.0476, 0)$, a repair row becomes $(0.0455, 0.9091, 0.0455)$, and a non-functional row becomes $(0, 0.0476, 0.9524)$. The weighted training expansion preserves total weight one for every source row, and preprocessing is fitted before expansion.

This is an ordinally informed soft-target treatment, not a fuzzy rule engine or an assertion that pump condition is a measured continuous variable.

## Fully fuzzy components and votes

Fuzzy XGBoost shows a small standalone improvement, reaching 81.164% at 10% overlap versus 81.014% with hard labels. Every fuzzy Random Forest is weaker than its 80.591% hard-label counterpart. When both components are fuzzy, the best 5% vote reaches 81.397%, loses all five folds and worsens log loss from 0.46851 to 0.48969.

The best fully fuzzy vote gains 22 correct repair and 84 non-functional decisions but loses 214 functional decisions, for a net loss of 108 rows.

In [ ]:
display(result['candidate_summary'].loc[:, [
    'mean_accuracy',
    'functional_recall',
    'repair_recall',
    'non_functional_recall',
    'log_loss',
]])
display(result['blend_summary'].loc[:, [
    'effective_repair_share',
    'mean_accuracy',
    'accuracy_change',
    'fold_wins',
    'worst_fold_change',
    'repair_recall_change',
    'passes_gate',
]])

## One-fuzzy-component crosses

Because fuzzy XGBoost improves in isolation while fuzzy Random Forest does not, the screen crosses each fuzzy component with its accepted hard-label counterpart without refitting. The closest result is 2.5% fuzzy XGBoost plus hard Random Forest at **81.534%**. It loses 0.090 points and all five folds. It gets seven more repair and 18 more non-functional rows right but loses 68 functional rows, a net loss of 43.

The component-level fuzzy signal is therefore real but not complementary enough to improve the accepted ensemble.

In [ ]:
result['cross_summary'].loc[:, [
    'model_name',
    'mean_accuracy',
    'accuracy_change',
    'fold_wins',
    'worst_fold_change',
    'repair_recall',
    'log_loss',
    'passes_gate',
]]

## Decision

Retain crisp labels and the accepted hard-label 55% XGBoost / 45% Random Forest vote. No fuzzy candidate passed the accuracy and fold gate, so the labelled local test remained closed and no competition prediction was generated.

Stop tuning a single global overlap. A defensible future fuzzy treatment would need row-specific uncertainty from repeated inspection, annotator agreement, temporal changes or another evidence source rather than another $\lambda$ chosen on these same folds.

The complete membership tables, component evidence and reproduction command are in [`../reports/fuzzy-target-membership-screen.md`](../reports/fuzzy-target-membership-screen.md).